# Weather Forecasting: GRU vs. Dense Architecture

This notebook evaluates a Gated Recurrent Unit (GRU) against our existing Dense (MLP) architecture to determine the optimal structure for ESP32 TFLite Micro deployment.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, GRU, Flatten
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

I0000 00:00:1777765411.659604 1895835 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Load Dataset

In [2]:
df = pd.read_csv('historical_weather_data.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)
print(f'Loaded {len(df)} days of historical weather data.')

Loaded 1461 days of historical weather data.


## 3. Feature Engineering (3D Time-Series Lags)
We explicitly format the input data into a 3D tensor of shape `(Samples, Timesteps, Features)`.

In [ ]:
weather_cols = ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'et0_fao_evapotranspiration', 'shortwave_radiation_sum', 'soil_moisture_0_to_7cm']

# Create flat lags first to shift data
for col in weather_cols:
    for i in range(1, 4):
        df[f'{col}_lag_{i}'] = df[col].shift(i)

# Create Target variables (predicting TOMORROW)
df['target_precipitation'] = df['precipitation_sum'].shift(-1)
df['target_et0'] = df['et0_fao_evapotranspiration'].shift(-1)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# Build the 3D X array: shape (samples, timesteps, features)
# Timestep 0 = 3 days ago, Timestep 1 = 2 days ago, Timestep 2 = yesterday
X_3d = np.zeros((len(df), 3, len(weather_cols)))
for i, col in enumerate(weather_cols):
    X_3d[:, 0, i] = df[f'{col}_lag_3']
    X_3d[:, 1, i] = df[f'{col}_lag_2']
    X_3d[:, 2, i] = df[f'{col}_lag_1']

y_precip = df['target_precipitation'].values
y_et0 = df['target_et0'].values

print(f'Input Shape: {X_3d.shape}')


Input Shape: (1457, 3, 6)


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration,shortwave_radiation_sum,soil_moisture_0_to_7cm,temperature_2m_max_lag_1,temperature_2m_max_lag_2,temperature_2m_max_lag_3,...,et0_fao_evapotranspiration_lag_2,et0_fao_evapotranspiration_lag_3,shortwave_radiation_sum_lag_1,shortwave_radiation_sum_lag_2,shortwave_radiation_sum_lag_3,soil_moisture_0_to_7cm_lag_1,soil_moisture_0_to_7cm_lag_2,soil_moisture_0_to_7cm_lag_3,target_precipitation,target_et0
0,2022-01-04,24.2,9.8,0.0,4.85,25.46,0.322792,23.8,24.1,24.4,...,4.77,4.78,25.97,25.84,26.51,0.340333,0.358417,0.382750,0.5,4.43
1,2022-01-05,23.1,10.1,0.5,4.43,23.45,0.315208,24.2,23.8,24.1,...,4.80,4.77,25.46,25.97,25.84,0.322792,0.340333,0.358417,0.4,3.80
2,2022-01-06,22.1,9.2,0.4,3.80,21.19,0.313958,23.1,24.2,23.8,...,4.85,4.80,23.45,25.46,25.97,0.315208,0.322792,0.340333,1.3,4.37
3,2022-01-07,22.0,11.2,1.3,4.37,23.86,0.306208,22.1,23.1,24.2,...,4.43,4.85,21.19,23.45,25.46,0.313958,0.315208,0.322792,0.3,4.37
4,2022-01-08,22.6,11.6,0.3,4.37,24.13,0.300333,22.0,22.1,23.1,...,3.80,4.43,23.86,21.19,23.45,0.306208,0.313958,0.315208,0.0,4.59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1452,2025-12-26,25.4,10.4,0.0,4.78,24.78,0.248167,24.8,23.8,23.5,...,4.14,4.36,24.52,22.17,23.27,0.301792,0.274542,0.267583,2.7,4.42
1453,2025-12-27,26.3,12.6,2.7,4.42,22.65,0.214167,25.4,24.8,23.8,...,4.61,4.14,24.78,24.52,22.17,0.248167,0.301792,0.274542,13.9,3.53
1454,2025-12-28,23.4,14.6,13.9,3.53,19.53,0.254292,26.3,25.4,24.8,...,4.78,4.61,22.65,24.78,24.52,0.214167,0.248167,0.301792,3.7,3.90
1455,2025-12-29,23.8,12.9,3.7,3.90,20.89,0.319458,23.4,26.3,25.4,...,4.42,4.78,19.53,22.65,24.78,0.254292,0.214167,0.248167,9.1,3.99


## 4. Train/Test Split & Normalization

In [4]:
split_idx = int(len(df) * 0.8)

X_train, X_test = X_3d[:split_idx], X_3d[split_idx:]
y_train_precip, y_test_precip = y_precip[:split_idx], y_precip[split_idx:]
y_train_et0, y_test_et0 = y_et0[:split_idx], y_et0[split_idx:]

# Normalize inputs using standard scaling
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std[X_std == 0] = 1e-6 # prevent division by zero

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

np.save('X_mean_3d.npy', X_mean)
np.save('X_std_3d.npy', X_std)
print(f'Training on {len(X_train)} days, Testing on {len(X_test)} days.')


Training on 1165 days, Testing on 292 days.


## 5. Model Architectures

In [5]:
def create_dense_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Flatten(), # Flattens the 3x6 grid into 18 before processing
        Dense(16, activation='relu'),
        Dense(8, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def create_gru_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        GRU(16, activation='tanh'),
        Dense(8, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


## 6. Train Models

In [6]:
input_shape = (3, len(weather_cols))

print("Training Dense Models...")
dense_precip = create_dense_model(input_shape)
dense_precip.fit(X_train_scaled, y_train_precip, epochs=50, batch_size=16, verbose=0)
dense_et0 = create_dense_model(input_shape)
dense_et0.fit(X_train_scaled, y_train_et0, epochs=50, batch_size=16, verbose=0)

print("Training GRU Models...")
gru_precip = create_gru_model(input_shape)
gru_precip.fit(X_train_scaled, y_train_precip, epochs=50, batch_size=16, verbose=0)
gru_et0 = create_gru_model(input_shape)
gru_et0.fit(X_train_scaled, y_train_et0, epochs=50, batch_size=16, verbose=0)

print("Training complete.")


Training Dense Models...


Training GRU Models...


Training complete.


## 7. Performance Comparison

In [7]:
def evaluate_model(model, X_test, y_test, name):
    preds = model.predict(X_test, verbose=0).flatten()
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    return mae, r2

# Precipitation
dense_mae_p, dense_r2_p = evaluate_model(dense_precip, X_test_scaled, y_test_precip, "Dense Precip")
gru_mae_p, gru_r2_p = evaluate_model(gru_precip, X_test_scaled, y_test_precip, "GRU Precip")

# ET0
dense_mae_e, dense_r2_e = evaluate_model(dense_et0, X_test_scaled, y_test_et0, "Dense ET0")
gru_mae_e, gru_r2_e = evaluate_model(gru_et0, X_test_scaled, y_test_et0, "GRU ET0")

print("========== PRECIPITATION ==========")
print(f"Dense - MAE: {dense_mae_p:.3f} mm  | R²: {dense_r2_p:.3f}")
print(f"GRU   - MAE: {gru_mae_p:.3f} mm  | R²: {gru_r2_p:.3f}")
print("\n========== EVAPOTRANSPIRATION ==========")
print(f"Dense - MAE: {dense_mae_e:.3f} mm  | R²: {dense_r2_e:.3f}")
print(f"GRU   - MAE: {gru_mae_e:.3f} mm  | R²: {gru_r2_e:.3f}")


========== PRECIPITATION ==========
Dense - MAE: 2.771 mm  | R²: -0.040
GRU   - MAE: 2.664 mm  | R²: 0.008

========== EVAPOTRANSPIRATION ==========
Dense - MAE: 0.508 mm  | R²: 0.088
GRU   - MAE: 0.484 mm  | R²: 0.192
